In [36]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# ---- Hyperparameters ----
input_shape = (41, 62)
embedding_dim = 128
batch_size = 16
num_epochs = 1
learning_rate = 1e-3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- Dataset Preprocessing ----
def load_dataset(folder):
    data_by_class = {}
    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)
        if os.path.isdir(cls_path):
            data_by_class[cls] = []
            for file in os.listdir(cls_path):
                if file.endswith('fps30.csv'):
                    csv_path = os.path.join(cls_path, file)
                    array = pd.read_csv(csv_path, header=None).values.astype(np.float32)
                    data_by_class[cls].append(array)
    return data_by_class

# ---- Pair Dataset ----
class ContrastiveDataset(Dataset):
    def __init__(self, data_by_class, pairs_per_class=10):
        self.pairs = []
        classes = list(data_by_class.keys())

        # same class pairs
        for cls in classes:
            samples = data_by_class[cls]
            for _ in range(pairs_per_class // 2):
                if len(samples) < 2:
                    continue
                a, b = random.sample(samples, 2)
                self.pairs.append((a, b, 1))

        # different class pairs
        for _ in range(5 * pairs_per_class // 2 * len(classes)):
            cls1, cls2 = random.sample(classes, 2)
            if not data_by_class[cls1] or not data_by_class[cls2]:
                continue
            a = random.choice(data_by_class[cls1])
            b = random.choice(data_by_class[cls2])
            self.pairs.append((a, b, 0))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        a, b, label = self.pairs[idx]
        return (
            torch.tensor(a).unsqueeze(0),  # shape (1, 41, 62)
            torch.tensor(b).unsqueeze(0),
            torch.tensor([label], dtype=torch.float32)
        )

# ---- Model ----
class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(62, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, embedding_dim)
        )

    def forward_once(self, x):
        # x: (B, 1, 41, 62)
        x = x.squeeze(1).permute(0, 2, 1)  # to (B, 62, 41)
        return self.encoder(x)

    def forward(self, x1, x2):
        e1 = self.forward_once(x1)
        e2 = self.forward_once(x2)
        return e1, e2

# ---- Contrastive Loss ----
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, label):
        dist = torch.norm(out1 - out2, p=2, dim=1)
        loss = label.squeeze() * dist**2 + (1 - label.squeeze()) * torch.clamp(self.margin - dist, min=0)**2
        return loss.mean()

# ---- Main Training ----

print("Loading dataset...")
data_by_class = load_dataset("modified_self_dataset_augmented")
dataset = ContrastiveDataset(data_by_class, pairs_per_class=20)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_set, test_set = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size)

model = SiameseNet().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Training...")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for a, b, label in train_loader:
        a, b, label = a.to(device), b.to(device), label.to(device)
        out1, out2 = model(a, b)
        loss = criterion(out1, out2, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {total_loss:.4f}")

# ---- Evaluation ----
print("Evaluating on test set...")
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for a, b, label in test_loader:
        a, b, label = a.to(device), b.to(device), label.to(device)
        out1, out2 = model(a, b)
        dist = torch.norm(out1 - out2, dim=1)
        preds = (dist < 0.5).float()  # threshold
        correct += (preds == label.squeeze()).sum().item()
        total += len(label)
print(f"Test Accuracy: {correct}/{total} = {100 * correct / total:.2f}%")

# ---- Save the model ----
save_path = "siamese_model.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")





Loading dataset...
Training...
Epoch 1/1 — Loss: 16.0593
Evaluating on test set...
Test Accuracy: 212/276 = 76.81%
Model saved to siamese_model.pth
